           Architecture:    
             
                 Upload Documents
                        │
        ┌───────────────┼───────────────┐
        │               │               │
       PDF            DOCX            TXT
        │               │               │
        └───────────────┼───────────────┘
                        │
                Document Loader
                        │
                        ▼
              Metadata Extraction
                        │
                        ▼
                  Text Chunking
                        │
                        ▼
               Embedding Generation
                        │
                        ▼
                   FAISS Database
                        │
                        ▼
                 Semantic Retrieval
                        │
                        ▼
                Large Language Model
                        │
        ┌───────────────┼───────────────┐
        ▼               ▼               ▼
 Literature Review    Compare Docs     Research Insights



In [8]:
"""
Document Loader Module

Loads supported documents from a folder and returns
their contents using LangChain document loaders.
"""

from pathlib import Path

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    UnstructuredMarkdownLoader,
    Docx2txtLoader
)


class DocumentLoader:

    def __init__(self):

        self.supported_formats = {
            ".pdf": PyPDFLoader,
            ".docx": Docx2txtLoader,
            ".txt": TextLoader,
            ".md": UnstructuredMarkdownLoader
        }

    def find_documents(self, folder_path):

        folder = Path(folder_path)

        if not folder.exists():
            raise FileNotFoundError(
                f"Folder '{folder}' does not exist."
            )

        files = []

        for file in folder.iterdir():

            if (
                file.is_file()
                and file.suffix.lower() in self.supported_formats
            ):
                files.append(file)

        return files

    def load_document(self, file_path):

        extension = Path(file_path).suffix.lower()

        if extension not in self.supported_formats:
            raise ValueError(
                f"Unsupported file format : {extension}"
            )

        loader_class = self.supported_formats[extension]

        loader = loader_class(str(file_path))

        return loader.load()

    def load_documents(self, folder_path):

        all_documents = []

        files = self.find_documents(folder_path)

        for file in files:

            docs = self.load_document(file)

            all_documents.extend(docs)

        return all_documents

In [4]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from app.rag.embedding_model import EmbeddingModel

embedding_model = EmbeddingModel()

text = "Retrieval Augmented Generation is an AI technique."

embedding = embedding_model.generate_embedding(text)

print("Embedding Dimension:", embedding_model.embedding_dimension())

print("Vector Shape:", embedding.shape)

print("\nFirst 10 values:")

print(embedding[:10])

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.
Embedding Dimension: 384
Vector Shape: (384,)

First 10 values:
[-0.08518542 -0.03616239 -0.05195662  0.05336625 -0.03954446  0.0737259
  0.03795977 -0.05291829 -0.00750555 -0.04133936]


c:\Users\RENIL DHOLARIYA\OneDrive\Desktop\PaperLensAI\app\rag\embedding_model.py:42: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


In [6]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from app.document_loaders.document_loader import DocumentLoader
from app.rag.text_splitter import TextSplitter
from app.rag.vector_store import VectorStore

loader = DocumentLoader()
splitter = TextSplitter()
store = VectorStore()

files = loader.find_documents("../data/uploads")

texts = []

for file in files:
    texts.append(loader.load_document(file))

chunks = splitter.split_documents(texts)

vector_db = store.create_vector_store(chunks)

print("Vector Store Created Successfully!")
print("Total Chunks Stored:", len(chunks))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector Store Created Successfully!
Total Chunks Stored: 226


In [7]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from app.document_loaders.document_loader import DocumentLoader
from app.rag.text_splitter import TextSplitter
from app.rag.vector_store import VectorStore
from app.rag.retriever import Retriever

loader = DocumentLoader()
splitter = TextSplitter()
store = VectorStore()

uploads = project_root / "data" / "uploads"

files = loader.find_documents(uploads)

texts = []

for file in files:
    texts.append(loader.load_document(file))

chunks = splitter.split_documents(texts)

vector_db = store.create_vector_store(chunks)

retriever = Retriever(vector_db)

results = retriever.retrieve(
    "What is Retrieval Augmented Generation?"
)

print(f"Retrieved {len(results)} chunks\n")

for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print("-" * 50)
    print(doc.page_content[:300])
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieved 5 chunks

Result 1
--------------------------------------------------
scoring functions between Retriever and Generator using KL
divergence.
V. AUGMENTATION PROCESS IN RAG
In the domain of RAG, the standard practice often involves
a singular (once) retrieval step followed by generation, which
can lead to inefficiencies and sometimes is typically insuffi-
cient for com

Result 2
--------------------------------------------------
llm evaluation of rag applications,” https://www.databricks.com/blog/
LLM-auto-eval-best-practices-RAG, 2023.
[164] S. Es, J. James, L. Espinosa-Anke, and S. Schockaert, “Ragas: Au-
tomated evaluation of retrieval augmented generation,” arXiv preprint
arXiv:2309.15217, 2023.
[165] J. Saad-Falcon, O.

Result 3
--------------------------------------------------
[14] Z. Shao, Y. Gong, Y. Shen, M. Huang, N. Duan, and W. Chen,
“Enhancing retrieval-augmented large language models with iterative
retrieval-generation synergy,” arXiv preprint arXiv:2305.15294, 